# nb124 — Cliff transformation mining (UNBOUNDED, Kaggle)

Run nb220 with **no MAX_TRANSFORMS cap and no MAX_RGS cap** on Kaggle to fully enumerate matched molecular pair transformations.

Pipeline:
1. MMP-fragment all 4,139 PXR train + 11,496 external NR compounds via `rdMMPA.FragmentMol`
2. Build R-group transformation prior: for every (R_A, R_B) pair, the empirical mean ΔpEC50 across all cores in which they appear
3. For test compounds, find core matches with training compounds, apply the transformation prior, produce per-compound expected pEC50 from analog + Δ
4. Use as augmented feature in scaffold-CV LGBM, blend with nb197

Kaggle has 30 GB RAM (vs ~16 GB locally), so unbounded enumeration should be feasible.

In [ ]:
import os, sys, time, warnings
os.environ['PYTHONIOENCODING'] = 'utf-8'
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import defaultdict
from rdkit import Chem
from rdkit.Chem import rdMMPA, AllChem
import lightgbm as lgb
from scipy.stats import spearmanr

from pxr.data import load_train, load_test
from pxr.chem import add_standard_columns
from pxr.eval import rae, scaffold_kfold_indices
from pxr.featurize import combined, impute
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

print('Setup OK')

In [ ]:
COLLAPSE_THRESH = 0.58
LGBM_BASE = dict(
    n_estimators=1500, num_leaves=63, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=10,
    objective='mae', n_jobs=-1, random_state=42, verbose=-1,
)

tr = load_train(); te_df = load_test()
tr = add_standard_columns(tr)
y_tr = tr['pec50'].values.astype(np.float64)
smiles_tr = tr['std_smiles'].tolist()
smiles_te = te_df['smiles'].tolist()
print(f'Train: {len(tr)}  Test: {len(te_df)}')

In [ ]:
def get_mmp_transforms(df, val_col='pec50'):
    '''UNBOUNDED MMP transform extraction (no MAX_RGS, no MAX_TRANSFORMS).'''
    print(f'  Fragmenting {len(df):,} compounds...')
    core_to_rgroups = defaultdict(list)
    t0 = time.time()
    for idx, row in df.iterrows():
        try:
            mol = Chem.MolFromSmiles(row['smiles'])
            if mol is None: continue
            frags = rdMMPA.FragmentMol(mol, resultsAsMols=False)
        except Exception:
            continue
        for frag in frags:
            if not isinstance(frag, tuple) or len(frag) != 2:
                continue
            core, rg = frag
            if rg:
                core_key = core if core else '*TERMINAL*'
                core_to_rgroups[core_key].append((row['smiles'], rg, row[val_col]))
        if idx % 1000 == 0 and idx > 0:
            print(f'    {idx}/{len(df)} ({time.time()-t0:.0f}s)')
    print(f'  Distinct cores: {len(core_to_rgroups):,}  ({time.time()-t0:.0f}s)')

    # UNBOUNDED pair enumeration
    transforms = []
    for core, rgs in core_to_rgroups.items():
        if core == '*TERMINAL*' or len(rgs) < 2:
            continue
        for i in range(len(rgs)):
            for j in range(i+1, len(rgs)):
                smi_a, rg_a, v_a = rgs[i]
                smi_b, rg_b, v_b = rgs[j]
                if rg_a == rg_b: continue
                transforms.append((core, rg_a, rg_b, v_a, v_b, v_b - v_a))
    print(f'  Distinct MMP transforms: {len(transforms):,}')
    return transforms, core_to_rgroups


def build_transform_prior(transforms, min_count=3):
    rg_pair_deltas = defaultdict(list)
    for core, ra, rb, va, vb, d in transforms:
        key = tuple(sorted([ra, rb]))
        sign = 1 if ra <= rb else -1
        rg_pair_deltas[key].append(d * sign)
    prior = {}
    for key, deltas in rg_pair_deltas.items():
        if len(deltas) >= min_count:
            prior[key] = {
                'mean': float(np.mean(deltas)),
                'median': float(np.median(deltas)),
                'std': float(np.std(deltas)),
                'count': len(deltas),
            }
    return prior

print('Functions defined')

In [ ]:
# --- A. Train MMPs ---
print('[A] Train MMPs (unbounded)')
train_simple = tr[['std_smiles', 'pec50']].rename(columns={'std_smiles': 'smiles'})
train_transforms, train_core_index = get_mmp_transforms(train_simple)
prior_train = build_transform_prior(train_transforms, min_count=3)
print(f'  Train prior covers {len(prior_train):,} R-group pairs (n>=3)')

In [ ]:
# --- B. External NR MMPs ---
import os
ext_path = '/kaggle/input/datasets/knowledgegraphlover/pxr-challenge-data/external/chembl_nr_extended.parquet'
if not os.path.exists(ext_path):
    # try local fallback
    for cand in ['data/external/chembl_nr_extended.parquet',
                 'external/chembl_nr_extended.parquet']:
        if os.path.exists(cand):
            ext_path = cand; break
if os.path.exists(ext_path):
    nr = pd.read_parquet(ext_path)
    nr_simple = nr[['std_smiles', 'pec50']].dropna().rename(columns={'std_smiles': 'smiles'})
    print(f'[B] External NR: {len(nr_simple):,} compounds')
    nr_transforms, _ = get_mmp_transforms(nr_simple)
    prior_nr = build_transform_prior(nr_transforms, min_count=3)
    print(f'  NR prior covers {len(prior_nr):,} R-group pairs')
else:
    print('External NR file not found; skipping')
    prior_nr = {}

In [ ]:
# --- C. Merge priors, weighted (train=3x external) ---
combined_prior = dict(prior_train)
for k, v in prior_nr.items():
    if k not in combined_prior:
        combined_prior[k] = v
    else:
        n_tr = combined_prior[k]['count']
        n_ex = v['count']
        combined_prior[k] = {
            'mean': (3*n_tr*combined_prior[k]['mean'] + n_ex*v['mean']) / (3*n_tr + n_ex),
            'median': combined_prior[k]['median'],
            'std': combined_prior[k]['std'],
            'count': n_tr + n_ex,
        }
print(f'[C] Merged prior: {len(combined_prior):,} R-group pairs')

In [ ]:
# --- D. Compute transform features for compounds ---
def compute_transform_feature(query_smiles, train_core_index, prior, label='compounds'):
    print(f'  [{label}] Computing features for {len(query_smiles)}...')
    preds = np.full(len(query_smiles), np.nan)
    deltas = np.zeros(len(query_smiles))
    n_matched = 0
    for i, smi in enumerate(query_smiles):
        mol = Chem.MolFromSmiles(smi)
        if mol is None: continue
        try:
            frags = rdMMPA.FragmentMol(mol, resultsAsMols=False)
        except Exception:
            continue
        anchors, ds = [], []
        for frag in frags:
            if not isinstance(frag, tuple) or len(frag) != 2: continue
            core, rg_q = frag
            if not rg_q: continue
            core_key = core if core else '*TERMINAL*'
            analogs = train_core_index.get(core_key, [])
            for tr_smi, tr_rg, tr_val in analogs:
                if tr_rg == rg_q:
                    anchors.append(tr_val); ds.append(0.0)
                else:
                    key = tuple(sorted([rg_q, tr_rg]))
                    if key in prior:
                        sign = 1 if tr_rg <= rg_q else -1
                        anchors.append(tr_val)
                        ds.append(prior[key]['median'] * sign)
        if anchors:
            anchors = np.array(anchors); ds = np.array(ds)
            preds[i] = float(np.median(anchors + ds))
            deltas[i] = float(np.median(ds))
            n_matched += 1
    print(f'  [{label}] {n_matched}/{len(query_smiles)} matched')
    return preds, deltas

test_pred, test_delta = compute_transform_feature(smiles_te, train_core_index, combined_prior, 'test')
finite = np.isfinite(test_pred)
print(f'Test transform predictions: {finite.sum()} valid')
if finite.sum() > 0:
    print(f'  range [{test_pred[finite].min():.2f}, {test_pred[finite].max():.2f}]  mean ± std: {test_pred[finite].mean():.2f} ± {test_pred[finite].std():.2f}')

In [ ]:
# --- E. Per-fold OOF transform predictions on train ---
scaffolds = tr['scaffold'].tolist()
folds = scaffold_kfold_indices(scaffolds, n_splits=5)
oof_pred = np.full(len(y_tr), np.nan)
oof_delta = np.zeros(len(y_tr))
for fold_idx, (tr_idx, va_idx) in enumerate(folds):
    print(f'\nFold {fold_idx+1}/5')
    fold_tr = train_simple.iloc[tr_idx].reset_index(drop=True)
    fold_transforms, fold_core_index = get_mmp_transforms(fold_tr)
    fold_prior = build_transform_prior(fold_transforms, min_count=3)
    merged = dict(fold_prior)
    for k, v in prior_nr.items():
        if k not in merged: merged[k] = v
    va_smiles = train_simple.iloc[va_idx]['smiles'].tolist()
    p, d = compute_transform_feature(va_smiles, fold_core_index, merged, f'fold{fold_idx+1}_val')
    oof_pred[va_idx] = p
    oof_delta[va_idx] = d

finite_oof = np.isfinite(oof_pred)
print(f'\nOOF transform coverage: {finite_oof.sum()}/{len(y_tr)} ({finite_oof.mean()*100:.1f}%)')
if finite_oof.sum() > 100:
    rho = spearmanr(y_tr[finite_oof], oof_pred[finite_oof]).correlation
    mae_t = np.mean(np.abs(y_tr[finite_oof] - oof_pred[finite_oof]))
    print(f'  Spearman rho = {rho:.3f}  MAE = {mae_t:.4f}')

In [ ]:
# --- F. Augmented LGBM ---
X_tr_base = combined(smiles_tr); X_tr_base = impute(X_tr_base)
X_te_base = combined(smiles_te); X_te_base = impute(X_te_base)
oof_filled = np.where(np.isfinite(oof_pred), oof_pred, np.nanmedian(oof_pred))
te_filled  = np.where(np.isfinite(test_pred), test_pred, np.nanmedian(oof_pred))
X_tr_aug = np.column_stack([X_tr_base, oof_filled, oof_delta])
X_te_aug = np.column_stack([X_te_base, te_filled, test_delta])
print(f'Augmented feature shape: train={X_tr_aug.shape}  test={X_te_aug.shape}')

results = {}
for name, X_tr_use, X_te_use in [('base_only', X_tr_base, X_te_base), ('cliff_aug', X_tr_aug, X_te_aug)]:
    oof = np.zeros(len(y_tr)); te_preds = []
    for tr_idx, va_idx in folds:
        m = lgb.LGBMRegressor(**LGBM_BASE)
        m.fit(X_tr_use[tr_idx], y_tr[tr_idx],
              eval_set=[(X_tr_use[va_idx], y_tr[va_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict(X_tr_use[va_idx])
        te_preds.append(m.predict(X_te_use))
    te_pred = np.mean(te_preds, axis=0)
    r = rae(y_tr, oof); ratio = te_pred.std() / oof.std()
    print(f'  {name}: OOF={r:.4f}  ratio={ratio:.3f}')
    results[name] = (oof, te_pred, r, ratio)

In [ ]:
# --- G. Save outputs ---
from pathlib import Path
out = Path('/kaggle/working/processed')
out.mkdir(exist_ok=True, parents=True)
import json

# Save the prior for downstream analysis
with open(out / 'cliff_transform_prior.json', 'w') as f:
    json.dump({'|'.join(k): v for k, v in combined_prior.items()}, f, indent=2)

# Save predictions
oof_aug, te_aug, r_aug, ratio_aug = results['cliff_aug']
np.save(out / 'oof_nb124_cliff_aug.npy', oof_aug)
np.save(out / 'te_nb124_cliff_aug.npy', te_aug)
np.save(out / 'oof_nb124_transform_pred.npy', oof_pred)
np.save(out / 'te_nb124_transform_pred.npy', test_pred)

sub = pd.DataFrame({'Molecule Name': te_df['name'], 'pEC50': te_aug})
sub_dir = Path('/kaggle/working/submissions')
sub_dir.mkdir(exist_ok=True, parents=True)
sub.to_csv(sub_dir / '124_cliff_aug.csv', index=False)

print(f'Saved cliff_aug OOF={r_aug:.4f}  ratio={ratio_aug:.3f}')
print('Done.')